# Set-up


In [200]:
import platform
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import rdkit
from IPython.display import Markdown, display
from rdkit import Chem
import requests
import json

import match_plantmetwiki as m

# Converting Inchikeys to Smiles


In [201]:
def key_to_smiles(inchikey):
    """
    """
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/%s/property/IsomericSMILES/TXT" % inchikey
    response = requests.get(url, timeout=30)
    smiles = response.text.strip()
    if smiles.startswith('Status: 404'):
        smiles = None
    return smiles

In [ ]:
def smiles_to_NPCs(smiles):
    if smiles != None:
            url = f"https://npclassifier.gnps2.org/classify?smiles=%s" %smiles
            response = requests.get(url, timeout = 30)
            if response.status_code != 200:
                             path = None
                             super = None
                             sub = None  
            else: 
                pathway = response.json()["pathway_results"]
                superclass = response.json()["superclass_results"]
                subclass = response.json()["class_results"]

                path = pathway[0] if len(pathway) == 1 else None
                super = superclass[0] if len(superclass) == 1 else None
                sub = subclass[0] if len(subclass) == 1 else None  
    else:
         path = None
         super = None
         sub = None  
    return path, super, sub

In [208]:
df_plant_met_wiki = pd.read_csv("./data/plantmetwiki_metabolites.csv")
InChiKey_full = df_plant_met_wiki[:10]
InChiKey_full.head()

# #  convert Inchikey To smiles
# for Key in InChiKey_full:
#     smiles = key_to_smiles(Key)
#     NPC =smiles_to_NPCs(smiles)
#     print(NPC)

InChiKey_full["smiles"] = InChiKey_full["inchikey"].apply(key_to_smiles)
InChiKey_full.columns



Index(['inchikey', 'name', 'wikidata', 'n_pathways', 'smiles'], dtype='str')

In [211]:
# counting the NaN, these Inchikeys have no smiles linked in Pubchem
no_smiles = InChiKey_full["smiles"].isna().sum()
print(no_smiles)


0


In [ ]:
subset_inchi = InChiKey_full[:20]
print(subset_inchi.columns)

# smile = "C1=CC=C2C(=C1)C(=C(N2)C3=NC4=CC=CC=C4C3=O)O"
# x, y, z = smiles_to_NPCs(smile)

subset_inchi["ncp_classification"] = subset_inchi["smiles"].apply(smiles_to_NPCs)

Index(['inchikey', 'name', 'wikidata', 'n_pathways', 'smiles'], dtype='str')
Index(['inchikey', 'name', 'wikidata', 'n_pathways', 'smiles',
       'ncp_classification'],
      dtype='str')


In [ ]:
def split_ncp_class(ncp_classification):
    path, super, sub = ncp_classification